# A2: U-Net DDPM vs DiT thu nhỏ trên CelebA

A2.1 đã nộp DeepLabv3. Nhóm đang đổi đề tài, chưa xác nhận giảng viên đồng ý. Đây là thí nghiệm pixel diffusion giản lược, không tái hiện đầy đủ latent DiT gốc. Chưa có kết quả train CelebA. AI hỗ trợ viết và kiểm tra code; nhóm tự kiểm chứng số liệu.

## 1. Drive, GPU và phụ thuộc

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, torch

drive.mount('/content/drive')
work_dir = Path('/content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/SDH_HK261_CO5085/A2')
os.chdir(work_dir)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 2. CelebA là gì?

CelebA gồm 202.599 ảnh khuôn mặt của 10.177 người, mỗi ảnh có 40 thuộc tính. Bài này chỉ học ảnh RGB để sinh khuôn mặt mới. Tải `img_align_celeba.zip` và `list_eval_partition.txt` qua [trang chính thức](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html), dùng cho nghiên cứu phi thương mại. Không dùng mã Drive chưa kiểm chứng. Giải nén cục bộ để giảm nghẽn I/O.

In [ ]:
import zipfile
CELEBA_ZIP = Path('/content/drive/MyDrive/datasets/img_align_celeba.zip')
PARTITION = Path('/content/drive/MyDrive/datasets/list_eval_partition.txt')
assert CELEBA_ZIP.is_file(), f'Chưa có {CELEBA_ZIP}'
assert PARTITION.is_file(), f'Chưa có {PARTITION}'
DATA = Path('/content/data/celeba')
DATA.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CELEBA_ZIP) as archive:
    archive.extractall(DATA)
print('Số ảnh:', sum(1 for _ in DATA.rglob('*.jpg')))


## 3. Kiểm tra dữ liệu và code

Split chính thức: 0 train, 1 val, 2 test. Validation/test không flip ngẫu nhiên. Smoke test dùng ảnh tổng hợp, không đo chất lượng sinh ảnh.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
from src.dataset import SimpleImageDataset
from torchvision.utils import make_grid
from IPython.display import display
from torchvision.transforms.functional import to_pil_image
for split in ['train', 'val', 'test']:
    dataset = SimpleImageDataset(DATA, split=split, partition_file=PARTITION)
    print(split, len(dataset))
dataset = SimpleImageDataset(DATA, split='val', partition_file=PARTITION)
display(to_pil_image(make_grid(torch.stack([dataset[i] for i in range(16)]), nrow=4)/2+.5))


## 4. Huấn luyện hai biến thể

Dùng cùng split, seed, resolution, số bước nhiễu, epoch và batch. Chọn checkpoint best theo val MSE. Mỗi lần train mới đặt RUN khác. Sampling 1.000 bước và train cả CelebA có thể lâu; không cam kết hoàn thành trong một phiên Colab.

In [ ]:
RUN = Path('runs/celeba_v1')
EPOCHS = 30
BATCH_SIZE = 16
for model in ['unet', 'dit']:
    subprocess.run([sys.executable, 'train.py', '--model', model,
                    '--data_dir', str(DATA), '--partition_file', str(PARTITION),
                    '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE),
                    '--seed', '42', '--save_dir', str(RUN/model),
                    '--sample_dir', str(RUN/model/'samples')], check=True)


## 5. Xem loss và tiến trình khử nhiễu

In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import Image
for model in ['unet', 'dit']:
    with (RUN/model/f'{model}_history.csv').open() as stream:
        rows = list(csv.DictReader(stream))
    for key in ['train_loss', 'val_loss']:
        plt.plot([int(r['epoch']) for r in rows], [float(r[key]) for r in rows], label=f'{model} {key}')
plt.xlabel('Epoch'); plt.ylabel('Noise MSE'); plt.legend(); plt.show()
for model in ['unet', 'dit']:
    output = RUN/model/'denoising.png'
    subprocess.run([sys.executable, 'sample.py', '--ckpt', str(RUN/model/f'{model}_best.pt'),
                    '--save_steps', '--num_images', '4', '--batch_size', '4', '--output', str(output)], check=True)
    display(Image(filename=str(output)))


## 6. FID trên test

Sinh từng ảnh riêng theo batch nhỏ, so sánh 5.000 ảnh thật test đã resize giống train. Dùng cùng seed và số ảnh cho hai mô hình. Không dùng ảnh ghép hoặc tập train làm ảnh thật đánh giá. Thư mục xuất cần trống. Lần đầu thư viện tải trọng số Inception; bước này có thể mất nhiều thời gian.

In [ ]:
N = 5000
for model in ['unet', 'dit']:
    output = RUN/model
    subprocess.run([sys.executable, 'sample.py', '--ckpt', str(output/f'{model}_best.pt'),
                    '--num_images', str(N), '--batch_size', '4', '--seed', '123',
                    '--output_dir', str(output/'generated')], check=True)
    subprocess.run([sys.executable, 'evaluate.py', '--data_dir', str(DATA),
                    '--partition_file', str(PARTITION), '--generated_dir', str(output/'generated'),
                    '--real_dir', str(output/'real_test'), '--num_images', str(N),
                    '--output', str(output/'fid.json')], check=True)
    print((output/'fid.json').read_text())


## 7. Nội dung cần đưa vào báo cáo

Báo số ảnh/split, cấu hình, thiết bị, train/val loss, FID 5k, số tham số, thời gian epoch và sampling (có I/O). Phân tích patchify, adaLN-Zero, epsilon-MSE; so sánh với latent/class-conditional DiT gốc. Không kết luận mô hình nào tốt hơn trước khi có số liệu. Bổ sung failure cases, hạn chế tính toán, PDF và video demo. [Google Scholar](https://scholar.google.com/scholar?q=Scalable+Diffusion+Models+with+Transformers).